Task 1: Basic Joins (INNER and LEFT)

In [29]:
import os

if os.path.exists("education.db"):
    os.remove("education.db")

In [30]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("education.db")
cursor = conn.cursor()

In [31]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS students (
student_id INTEGER PRIMARY KEY,
name TEXT,
email TEXT,
city TEXT
)
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS enrollments (
enrollment_id INTEGER PRIMARY KEY,
student_id INTEGER,
course_name TEXT,
enrollment_date TEXT
)
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS grades (
grade_id INTEGER PRIMARY KEY,
student_id INTEGER,
course_name TEXT,
score INTEGER
)
""")

conn.commit()

In [32]:
#Students
students = [
(1,'Alice','alice@email.com','Mumbai'),
(2,'Bob','bob@email.com','Delhi'),
(3,'Carol','carol@email.com','Bangalore'),
(4,'David','david@email.com','Chennai'),
(5,'Emma','emma@email.com','Pune')
]

cursor.executemany("INSERT INTO students VALUES (?,?,?,?)", students)

In [33]:
#Students
enrollments = [
(101,1,'Python','2024-01-15'),
(102,1,'SQL','2024-02-01'),
(103,2,'Data Science','2024-01-20'),
(104,3,'Python','2024-02-10'),
(105,5,'Machine Learning','2024-01-25')
]

cursor.executemany("INSERT INTO enrollments VALUES (?,?,?,?)", enrollments)

In [34]:
#Grades
grades = [
(201,1,'Python',85),
(202,1,'SQL',90),
(203,2,'Data Science',78),
(204,4,'Web Development',88)
]

cursor.executemany("INSERT INTO grades VALUES (?,?,?,?)", grades)

conn.commit()

In [35]:
#Inner join
query = """
SELECT s.name, e.course_name
FROM students AS s
INNER JOIN enrollments AS e
ON s.student_id = e.student_id
"""
df = pd.read_sql_query(query, conn)
df

,name,course_name
0,Alice,Python
1,Alice,SQL
2,Bob,Data Science
3,Carol,Python
4,Emma,Machine Learning


In [36]:
#Left Join
query = """
SELECT s.name, e.course_name
FROM students AS s
LEFT JOIN enrollments AS e
ON s.student_id = e.student_id
"""

df = pd.read_sql_query(query, conn)
df

,name,course_name
0,Alice,Python
1,Alice,SQL
2,Bob,Data Science
3,Carol,Python
4,David,None
5,Emma,Machine Learning


Explanation:

LEFT JOIN returns all records from the left table (students) and the matching records from the right table (enrollments).

If a student has no enrollment, the course_name will appear as NULL.

| name  | course_name      |
| ----- | ---------------- |
| Alice | Python           |
| Alice | SQL              |
| Bob   | Data Science     |
| Carol | Python           |
| David | NULL             |
| Emma  | Machine Learning |

Here David appears with NULL because he has no enrollment.

Why do we use table aliases like AS s and AS e? Rewrite Query 1 without aliases and explain why aliases are helpful.

Query 1 Without Aliases

SELECT students.name, enrollments.course_name
FROM students
INNER JOIN enrollments
ON students.student_id = enrollments.student_id;

Explanation:

Table aliases are short names given to tables in SQL queries. They make queries shorter and easier to read, especially when working with multiple tables.

Query 2 With Aliases

SELECT s.name, e.course_name
FROM students AS s
INNER JOIN enrollments AS e
ON s.student_id = e.student_id;

Explanation:

Here:

s represents the students table

e represents the enrollments table

Instead of writing long names like students.student_id, we can write s.student_id.

Task 2: Multiple Table Joins

In [37]:
query = """
SELECT s.name, e.course_name, g.score
FROM students AS s
LEFT JOIN enrollments AS e
ON s.student_id = e.student_id
LEFT JOIN grades AS g
ON s.student_id = g.student_id AND e.course_name = g.course_name
"""

df = pd.read_sql_query(query, conn)
df

,name,course_name,score
0,Alice,Python,85.0
1,Alice,SQL,90.0
2,Bob,Data Science,78.0
3,Carol,Python,NaN
4,David,None,NaN
5,Emma,Machine Learning,NaN


Explanation:

Why we use LEFT JOIN instead of INNER JOIN

LEFT JOIN is used to show all students, even if they have no enrollments or grades.

If INNER JOIN was used, only students with matching records in all tables would appear.

What happens to students with no enrollments

Students who are not enrolled in any course will still appear in the result.
However, the course_name and score will be NULL.

Example:

name	course_name	score
David	NULL	NULL

Example:

name	course_name	score
Carol	Python	NULL

Why the join condition includes both student_id AND course_name
ON s.student_id = g.student_id AND e.course_name = g.course_name

This ensures that the correct grade is matched with the correct course.

If only student_id was used, a student's grade might be linked to the wrong course.


Task 3: Understanding Joins and Normalization

1)Why normalize data? Explain why student information is split across three tables instead of one large table with all data repeated.

Normalization is the process of organizing data into multiple related tables to reduce redundancy and improve data consistency.

Instead of storing all information in one large table, the data is divided into separate tables such as:

students table → stores student details (name, email, city)

enrollments table → stores which courses students are enrolled in

grades table → stores the scores of students in courses

Benefits of Normalization

Reduces data duplication – student details are stored only once

Improves data integrity – changes need to be made in only one place

Better organization – data is logically separated

Efficient storage and maintenance

If all data were stored in one table, student information would repeat many times for each course, which wastes space and makes updates harder.

2)When to use each join type:
Use INNER JOIN when: ___
Use LEFT JOIN when: ___

Use INNER JOIN when:

You want only the records that exist in both tables.

Example:
Finding students who are enrolled in courses.

Students without enrollments will not appear in the result.

Use LEFT JOIN when:

You want all records from the left table, even if there is no matching record in the right table.

Example:
Showing all students, including those who have not enrolled in any course.

Missing values will appear as NULL.

3)Pandas equivalent: If you had these three tables as DataFrames (students_df, enrollments_df, grades_df), write the Pandas code to perform the same multiple join from Task 2:



In [38]:
students_df = pd.read_sql_query("SELECT * FROM students", conn)
enrollments_df = pd.read_sql_query("SELECT * FROM enrollments", conn)
grades_df = pd.read_sql_query("SELECT * FROM grades", conn)

students_df

,student_id,name,email,city
0,1,Alice,alice@email.com,Mumbai
1,2,Bob,bob@email.com,Delhi
2,3,Carol,carol@email.com,Bangalore
3,4,David,david@email.com,Chennai
4,5,Emma,emma@email.com,Pune


In [39]:
#3)Pandas equivalent: If you had these three tables as DataFrames (students_df, enrollments_df, grades_df), write the Pandas code to perform the same multiple join from Task 2:
import pandas as pd

# Merge students and enrollments
result = pd.merge(students_df, enrollments_df,
                  on='student_id', how='left')

# Merge the result with grades
result = pd.merge(result, grades_df,
                  on=['student_id', 'course_name'], how='left')

result

,student_id,name,email,city,enrollment_id,course_name,enrollment_date,grade_id,score
0,1,Alice,alice@email.com,Mumbai,101.0,Python,2024-01-15,201.0,85.0
1,1,Alice,alice@email.com,Mumbai,102.0,SQL,2024-02-01,202.0,90.0
2,2,Bob,bob@email.com,Delhi,103.0,Data Science,2024-01-20,203.0,78.0
3,3,Carol,carol@email.com,Bangalore,104.0,Python,2024-02-10,NaN,NaN
4,4,David,david@email.com,Chennai,NaN,NaN,NaN,NaN,NaN
5,5,Emma,emma@email.com,Pune,105.0,Machine Learning,2024-01-25,NaN,NaN


In [40]:
result = pd.merge(students_df, enrollments_df,
                  on='student_id', how='left')

result = pd.merge(result, grades_df,
                  on=['student_id','course_name'], how='left')

result

,student_id,name,email,city,enrollment_id,course_name,enrollment_date,grade_id,score
0,1,Alice,alice@email.com,Mumbai,101.0,Python,2024-01-15,201.0,85.0
1,1,Alice,alice@email.com,Mumbai,102.0,SQL,2024-02-01,202.0,90.0
2,2,Bob,bob@email.com,Delhi,103.0,Data Science,2024-01-20,203.0,78.0
3,3,Carol,carol@email.com,Bangalore,104.0,Python,2024-02-10,NaN,NaN
4,4,David,david@email.com,Chennai,NaN,NaN,NaN,NaN,NaN
5,5,Emma,emma@email.com,Pune,105.0,Machine Learning,2024-01-25,NaN,NaN
